# 05 · Test inference and submission

1. Stage-1 and stage-2 scores for every test candidate (average of the 5 fold models).
2. Frozen decision rule from `decision.json`.
3. Invariants asserted before writing: predicted ⊆ candidates, no record under two S1s, only known S1 ids; every test S1 gets exactly one row (empty lists allowed).
4. Official validator.
5. Submission zip in the required layout.

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
# Every stage call below is checkpointed: re-running a notebook skips finished work.
# Production runs don't need the notebooks: `bash scripts/run_notebooks.sh` (see README).
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Optional overrides (or export EF_* variables before starting Jupyter):
# os.environ["EF_DEV_MODE"] = "1"      # small consistent slice (artifacts_dev/)
# os.environ["EF_PROFILE"] = "32gb"    # 16gb | 32gb | 64gb (default: auto from RAM)
# os.environ["EF_N_THREADS"] = "4"

import entity_forge  # noqa: F401  (caps thread pools; must come before polars)
import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(40); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(S.describe())

In [ ]:
stages.run_predict_test(S)

In [ ]:
summary = stages.run_write_submission(S)
summary

## Sanity: predicted set sizes per country (France is unseen in training)

In [ ]:
from entity_forge.io import read_tsv, MATCH_HEADER
res = read_tsv(S.output_dir / "matching_results.tsv", MATCH_HEADER).with_columns(
    pl.col("matched_entity_ids").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.len().alias("n"))
s1c = pl.read_parquet(S.norm("test", 1), columns=["entity_id", "ckey"])
(res.join(s1c, left_on="source1_entity_id", right_on="entity_id")
    .group_by("ckey").agg((pl.col("n") == 0).mean().alias("empty_rate"), pl.col("n").mean().alias("avg_matches"), pl.len()))

Training reference: empty rate ≈ 5.6 %, average ≈ 3.46 matches per S1. Large deviations for France suggest a normalization gap.

In [ ]:
print(stages.run_validator(S, check_ids=True))

In [ ]:
zip_path = stages.build_submission_zip(S)
zip_path

## Checkpoint status

In [ ]:
pl.DataFrame(stages.status_rows(S))